# Pawn counting

Project to explore/visualise chess pawn structures

In [165]:
from typing import Callable

from pathlib import Path
import functools
import numpy as np
import polars as pl
import altair as alt
import scipy.sparse as sp
from sklearn.manifold import spectral_embedding
import polyscope as ps
import ipywidgets as widgets

In [166]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## basic functions

In [167]:
type Position = tuple[np.uint64, np.uint64]
"""
Pawn structure position --- a configuration of White and Black pawns on a board of size
BOARD_WIDTH * BOARD_DEPTH

Consists of two u64 bitmasks, the first for White pawns and the second for Black.

The least bit of each bitmask is the bottom-left square for White (a1), counting up in
file-major order (a1, a2, ..., a8, b1, ...) to the top-right square for White (h8).

If the board is smaller than 8x8, the board crops around bottom-left square for White
(so the bottom-left square remains a1), and the unused squares are empty.
"""

BOARD_WIDTH = 2
BOARD_DEPTH = 3

assert 0 <= BOARD_WIDTH <= 8
assert 0 <= BOARD_DEPTH <= 8

In [168]:
def position_to_int(pos: Position) -> int:
    """Pack a Position into a single UInt128: low 64 bits White, high 64 bits Black."""
    white, black = pos
    return int(white) | (int(black) << 64)


def position_from_int(code: int) -> Position:
    """Unpack a UInt128 (low bits White, high bits Black) back into a Position."""
    mask = (1 << 64) - 1
    return np.uint64(code & mask), np.uint64((code >> 64) & mask)

In [169]:
_CACHE = f"./transitions_{BOARD_WIDTH}x{BOARD_DEPTH}.tmp"


def cached_frame(name: str):
    def cached_frame_inner(func: Callable[..., pl.DataFrame]):
        nonlocal name

        @functools.wraps(func)
        def inner_func(*args, **kwargs):
            Path(_CACHE).mkdir(parents=True, exist_ok=True)

            cached_frame_path = Path(_CACHE) / f"{name}.parquet"
            if cached_frame_path.exists():
                return pl.read_parquet(cached_frame_path)

            frame = func(*args, **kwargs)
            frame.write_parquet(cached_frame_path)
            return frame

        return inner_func

    return cached_frame_inner


In [170]:
def position_ndarray(pos: Position) -> tuple[np.ndarray, np.ndarray]:
    def to_array(mask):
        arr = np.zeros((BOARD_WIDTH, BOARD_DEPTH), dtype=bool)
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                arr[f, r] = (mask >> (f * 8 + r)) & 1
        return arr

    white, black = pos
    return to_array(white), to_array(black)


def init_position() -> Position:
    white = 0
    black = 0
    for f in range(BOARD_WIDTH):
        white |= 1 << (f * 8)
        black |= 1 << (f * 8 + BOARD_DEPTH - 1)
    return np.uint64(white), np.uint64(black)


def rand_position(max_pawns_per_side: int | None = None) -> Position:
    squares = [f * 8 + r for f in range(BOARD_WIDTH) for r in range(BOARD_DEPTH)]
    n = len(squares)
    cap = n if max_pawns_per_side is None else min(max_pawns_per_side, n)

    n_white = np.random.randint(cap + 1)
    n_black = np.random.randint(min(cap, n - n_white) + 1)

    chosen = np.random.choice(n, size=n_white + n_black, replace=False)
    white = 0
    black = 0
    for i in chosen[:n_white]:
        white |= 1 << squares[i]
    for i in chosen[n_white:]:
        black |= 1 << squares[i]
    return np.uint64(white), np.uint64(black)


def pawns_as_frame(pos: Position) -> pl.DataFrame:
    colour = pl.Enum(["White", "Black"])
    rows = []
    for name, mask in zip(["White", "Black"], pos):
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                if (mask >> (f * 8 + r)) & 1:
                    rows.append({"rank": r + 1, "file": f + 1, "colour": name})
    return pl.DataFrame(
        rows, schema={"rank": pl.Int64, "file": pl.Int64, "colour": colour}
    )

In [171]:
def position_chart(pos: Position, *, axis=False, legend=False) -> alt.LayerChart:
    _PAWN_COLOURS = {
        "domain": ["White", "Black"],
        "range": ["white", "black"],
    }
    _SQUARE_COLOURS = {
        "domain": ["light", "dark"],
        "range": ["#f0d9b5", "#b58863"],
    }
    _FILE_DOMAIN = list(range(1, BOARD_WIDTH + 1))
    _RANK_DOMAIN = list(range(1, BOARD_DEPTH + 1))

    def _board_as_frame() -> pl.DataFrame:
        rows = []
        for f in range(1, BOARD_WIDTH + 1):
            for r in range(1, BOARD_DEPTH + 1):
                square = "dark" if (f + r) % 2 == 0 else "light"
                rows.append({"rank": r, "file": f, "square": square})
        return pl.DataFrame(rows)

    axis_params = {} if axis else {"axis": None}
    legend_params = {} if legend else {"legend": None}

    board = (
        alt.Chart(_board_as_frame())
        .mark_rect()
        .encode(
            alt.X("file:O", **axis_params).scale(domain=_FILE_DOMAIN),  # type: ignore
            alt.Y("rank:O", **axis_params).scale(domain=_RANK_DOMAIN),  # type: ignore
            alt.Color("square:N")  # type: ignore
            .scale(**_SQUARE_COLOURS)  # type: ignore
            .legend(None),
        )
    )

    pawns = (
        alt.Chart(pawns_as_frame(pos))
        .mark_circle(size=250, stroke="black", strokeWidth=0.5)
        .encode(
            alt.X("file:O").scale(domain=_FILE_DOMAIN),
            alt.Y("rank:O").scale(domain=_RANK_DOMAIN, reverse=True),
            alt.Color("colour:N", **legend_params)  # type: ignore
            #
            .scale(**_PAWN_COLOURS),  # type: ignore
        )
    )

    return (
        alt.layer(board, pawns)
        .resolve_scale(color="independent")
        .properties(width=33 * BOARD_WIDTH, height=33 * BOARD_DEPTH)
    )  # type: ignore


position_chart(init_position())

alt.LayerChart(...)

In [172]:
def _on_board(f: int, r: int) -> bool:
    return 0 <= f < BOARD_WIDTH and 0 <= r < BOARD_DEPTH


def accessible_positions(pos: Position) -> list[tuple[str, int, Position]]:
    white, black = int(pos[0]), int(pos[1])
    out: list[tuple[str, int, Position]] = []

    def emit(kind, src, new_own, new_enemy, white_to_move):
        w, b = (new_own, new_enemy) if white_to_move else (new_enemy, new_own)
        out.append((kind, src, (np.uint64(w), np.uint64(b))))

    for white_to_move in (True, False):
        own, enemy = (white, black) if white_to_move else (black, white)
        dr = 1 if white_to_move else -1
        for f in range(BOARD_WIDTH):
            for r in range(BOARD_DEPTH):
                src = f * 8 + r
                if not (own >> src) & 1:
                    continue
                own_removed = own & ~(1 << src)

                af, ar = f, r + dr  # advance (straight)
                if _on_board(af, ar):
                    dst = af * 8 + ar
                    if not ((own | enemy) >> dst) & 1:
                        emit("A", src, own_removed | (1 << dst), enemy, white_to_move)
                else:
                    emit(
                        "A", src, own_removed, enemy, white_to_move
                    )  # off end rank == R

                for kind, df in (("CQ", -1), ("CK", 1)):  # diagonal captures
                    cf, cr = f + df, r + dr
                    if not 0 <= cf < BOARD_WIDTH:
                        continue  # off the side: not a legal capture
                    if not 0 <= cr < BOARD_DEPTH:
                        emit(
                            kind, src, own_removed, enemy, white_to_move
                        )  # off end rank == R
                        continue
                    dst = cf * 8 + cr
                    if (own >> dst) & 1:
                        continue  # friendly block
                    emit(
                        kind,
                        src,
                        own_removed | (1 << dst),
                        enemy & ~(1 << dst),
                        white_to_move,
                    )

                emit("R", src, own_removed, enemy, white_to_move)  # always

    return out

In [173]:
def multiple_positions_chart(
    poses: list[Position], *, width: int | None = None
) -> alt.VConcatChart:
    pos_charts = [position_chart(p) for p in poses]
    if width is None:
        width = int(np.sqrt(len(pos_charts)))
    return alt.vconcat(
        *(
            alt.hconcat(*pos_charts[n : (n + width)])
            for n in range(0, len(pos_charts), width)
        )
    )


multiple_positions_chart(
    [p for _, _, p in accessible_positions(init_position())], width=7
)

alt.VConcatChart(...)

In [174]:
def button(description: str = "Run"):
    """Defer a side-effecting `func` (which returns None) behind an ipywidgets button.

    Mirrors `cached_frame`: use as a decorator, then call the wrapped function to get a
    Button instead of running `func`. Each click runs `func(*args, **kwargs)` with the
    args passed when the button was made. Display the button as the cell's last expression.
    """

    def button_inner(func: Callable[..., None]):
        @functools.wraps(func)
        def inner_func(*args, **kwargs):
            btn = widgets.Button(description=description)
            btn.on_click(lambda _clicked: func(*args, **kwargs))
            return btn

        return inner_func

    return button_inner


def display_coloured_mesh(
    nodes: np.ndarray,
    edges: np.ndarray,
    edge_colours: np.ndarray,
    *,
    name: str = "mesh",
    radius: float = 0.002,
    drop_unused_nodes: bool = True,
    normalise: bool = True,
    highlight_nodes: np.ndarray | None = None,
) -> ps.CurveNetwork:
    """Display `edges` as a polyscope line mesh, one RGB colour per edge.

    nodes:        (N, 3) float node positions.
    edges:        (M, 2) int index pairs into `nodes`.
    edge_colours: (M, 3) float RGB in [0, 1], row-aligned with `edges`.

    `drop_unused_nodes` reindexes onto only the nodes the edges touch, so the bounding
    box stays tight around what is actually drawn. `normalise` recentres and rescales
    each axis independently so the mesh fills a unit cube -- handy when coordinates are
    tiny or very anisotropic (as the spectral embedding's are); pass False to keep the
    true aspect ratio. `highlight_nodes` are node indices (into `nodes`) to mark with
    spheres; they get the same reindex/normalise transform so they line up with the mesh
    (indices dropped by `drop_unused_nodes` are skipped).
    """
    nodes = np.asarray(nodes, dtype=float)
    edges = np.asarray(edges)
    highlight = (
        None if highlight_nodes is None else np.asarray(highlight_nodes).reshape(-1)
    )

    if drop_unused_nodes:
        used = np.unique(edges)
        remap = np.full(len(nodes), -1, dtype=np.int64)
        remap[used] = np.arange(len(used))
        edges = remap[edges]
        nodes = nodes[used]
        if highlight is not None:
            highlight = remap[highlight]
            highlight = highlight[highlight >= 0]

    if normalise:
        nodes = nodes - nodes.mean(axis=0)
        span = np.abs(nodes).max(axis=0)
        span[span == 0] = 1.0
        nodes = nodes / span

    print(
        f"{len(edges)} edges, {len(nodes)} nodes; extents "
        f"{nodes.min(axis=0).round(3)} .. {nodes.max(axis=0).round(3)}"
    )

    ps.init()
    net = ps.register_curve_network(name, nodes, edges)
    net.set_radius(radius, relative=True)  # visible tube; bump if still too thin
    net.add_color_quantity(
        "colour", np.asarray(edge_colours), defined_on="edges", enabled=True
    )

    if highlight is not None and len(highlight):
        marker = ps.register_point_cloud(f"{name} (initial state)", nodes[highlight])
        marker.set_radius(0.01, relative=True)
        marker.set_color((1.0, 0.85, 0.0))

    ps.show()
    return net

## generation

In [175]:
def explore_transitions(start: Position) -> pl.DataFrame:
    """Every transition reachable from `start`, one row per transition.

    Positions are packed into a UInt128: low 64 bits White, high 64 bits Black.
    `transition_depth` is the minimum number of steps to reach the position the
    move is made from (the BFS level of `start_pos`).

    Edges are collected one BFS level at a time and concatenated, keeping peak
    memory bounded (~2.4 GB / ~2 min for the 4x4 start position: ~2.2M positions,
    ~46M transitions).
    """
    transition_type = pl.Enum(["A", "CQ", "CK", "R"])
    schema = {
        "start_pos": pl.UInt128,
        "end_pos": pl.UInt128,
        "moving_pawn": pl.UInt8,
        "transition_type": transition_type,
    }

    def encode(pos: Position) -> int:
        return int(pos[0]) | (int(pos[1]) << 64)

    seen = {encode(start)}
    frontier = [start]
    chunks: list[pl.DataFrame] = []

    depth = 0
    while frontier:
        starts, ends, pawns, kinds = [], [], [], []
        next_frontier = []
        for pos in frontier:
            start_code = encode(pos)
            for kind, src, end in accessible_positions(pos):
                end_code = encode(end)
                starts.append(start_code)
                ends.append(end_code)
                pawns.append(src)
                kinds.append(kind)
                if end_code not in seen:
                    seen.add(end_code)
                    next_frontier.append(end)
        if starts:
            chunks.append(
                pl.DataFrame(
                    {
                        "start_pos": starts,
                        "end_pos": ends,
                        "moving_pawn": pawns,
                        "transition_type": kinds,
                    },
                    schema=schema,
                ).with_columns(transition_depth=pl.lit(depth, dtype=pl.UInt8))
            )
        frontier = next_frontier
        depth += 1

    return pl.concat(chunks, rechunk=False)


@cached_frame("transitions")
def _transitions() -> pl.DataFrame:
    return explore_transitions(init_position())


transitions = _transitions()
print(transitions.schema)
transitions

Schema({'start_pos': UInt128, 'end_pos': UInt128, 'moving_pawn': UInt8, 'transition_type': Enum(categories=['A', 'CQ', 'CK', 'R']), 'transition_depth': UInt8})


start_pos,end_pos,moving_pawn,transition_type,transition_depth
u128,u128,u8,enum,u8
18963252907773419061505,18963252907773419061506,0,"""A""",0
18963252907773419061505,18963252907773419062016,0,"""CK""",0
18963252907773419061505,18963252907773419061504,0,"""R""",0
18963252907773419061505,18963252907773419061761,8,"""A""",0
18963252907773419061505,18963252907773419061251,8,"""CQ""",0
…,…,…,…,…
4740813226943354766340,4722366482869645214724,0,"""CK""",8
4740813226943354766340,4722366482869645214724,0,"""R""",8
4740813226943354766340,18446744073709552644,8,"""A""",8


In [176]:
def extract_positions(transitions: pl.DataFrame) -> pl.DataFrame:
    return (
        pl.concat(
            [
                transitions.lazy().select(pos="start_pos"),
                transitions.lazy().select(pos="end_pos"),
            ]
        )
        .select(pl.col("pos").unique(maintain_order=True))
        .collect()
    )


positions = extract_positions(transitions).with_row_index("position_id")
positions

position_id,pos
u32,u128
0,18963252907773419061505
1,18963252907773419061506
2,18963252907773419062016
3,18963252907773419061504
4,18963252907773419061761
…,…
276,4740813226943354766338
277,4740813226943354766848
278,4759259971017064317956


## exploration

### transition depth

In [177]:
# branching factor plot

_plot_df = (
    transitions.group_by("transition_depth", "start_pos")
    .agg(transitions_count=pl.len())
    .group_by("transition_depth")
    .agg(
        transitions_count=pl.col("transitions_count").sum(),
        branching_factor=pl.col("transitions_count").mean(),
    )
    .sort("transition_depth")
)

alt.hconcat(
    alt.Chart(_plot_df).mark_line().encode(x="transition_depth", y="transitions_count"),
    alt.Chart(_plot_df).mark_line().encode(x="transition_depth", y="branching_factor"),
)

alt.HConcatChart(...)

In [178]:
# positions leading to empty
multiple_positions_chart(
    [
        position_from_int(long_pos)
        for long_pos in transitions.filter(
            pl.col("end_pos") == 0, pl.col("transition_type") != "R"
        )["start_pos"]
    ],
    width=7,
)

alt.VConcatChart(...)

### spectral mesh visualisation

In [179]:
def positions_adjacency(
    transitions: pl.DataFrame,
    positions: pl.DataFrame,
    *,
    weight: pl.Expr = pl.lit(1.0),
) -> sp.csr_matrix:
    """Symmetric weighted adjacency of the position graph, indexed by `positions` row order.

    `weight` is a Polars expression evaluated against `transitions`, giving a weight per
    transition. Parallel transitions between the same pair are collapsed by taking the max
    weight; transitions are directed, so we symmetrise (undirected) by taking the max weight
    over the two directions, and drop self-loops. The default `weight=pl.lit(1.0)` reproduces
    the plain 0/1 adjacency.
    """
    nodes = positions
    edges = (
        transitions.select("start_pos", "end_pos", w=weight)
        .join(nodes.rename({"pos": "start_pos", "position_id": "i"}), on="start_pos")
        .join(nodes.rename({"pos": "end_pos", "position_id": "j"}), on="end_pos")
        .group_by("i", "j")
        .agg(pl.col("w").max())
    )
    i = edges["i"].to_numpy()
    j = edges["j"].to_numpy()
    w = edges["w"].to_numpy().astype(float)
    n = positions.height
    A = sp.coo_matrix((w, (i, j)), shape=(n, n)).tocsr()
    A = A.maximum(A.T)  # undirected: max weight over the two directions
    A.setdiag(0)  # no self-loops
    A.eliminate_zeros()
    return A  # type: ignore


def spectral_embedding_3d(
    transitions: pl.DataFrame,
    positions: pl.DataFrame,
    *,
    n_dims: int = 3,
    weight: pl.Expr = pl.lit(1.0),
    eigen_solver: str = "amg",
) -> pl.DataFrame:
    """Spectral embedding of the position graph: `positions` plus (x, y, z) columns.

    `n_dims` (1..3) is how many embedding coordinates to actually compute; the unused axes
    among (x, y, z) are filled with 0 so the output schema never changes. `weight` is a
    Polars expression over `transitions` giving each edge's weight (default `1.0`, unweighted).

    Rows line up with `positions`. Coordinates are the smallest non-trivial eigenvectors
    of the normalised graph Laplacian. `eigen_solver="amg"` (pyamg-preconditioned LOBPCG)
    scales to millions of nodes; "arpack" (shift-invert) does NOT here -- fill-in on this
    expander-like graph makes it blow past minutes even at 3x4. "lobpcg" is fine small.
    """
    if not 1 <= n_dims <= 3:
        raise ValueError(f"n_dims must be between 1 and 3, got {n_dims}")
    A = positions_adjacency(transitions, positions, weight=weight)
    emb = spectral_embedding(
        A,
        n_components=n_dims,
        eigen_solver=eigen_solver,  # type: ignore
        drop_first=True,
        random_state=0,
    )
    n = positions.height
    coords = [emb[:, k] if k < n_dims else np.zeros(n) for k in range(3)]
    return positions.with_columns(
        x=pl.Series(coords[0]), y=pl.Series(coords[1]), z=pl.Series(coords[2])
    )


spectral_embeddings = spectral_embedding_3d(
    transitions,
    positions,
    weight=pl.col("transition_type").replace_strict(
        {
            "A": 1 / 1.0,
            "CK": 1 / 1.0,
            "CQ": 1 / 1.0,
            "R": 1 / 1.0,
        }
    ),
)
spectral_embeddings

/home/nic/repos/movpasd/pawncounter/.venv/lib/python3.14/site-packages/sklearn/manifold/_spectral_embedding.py:430: UserWarning: Exited at iteration 20 with accuracies 
[1.79007241e-16 1.69394316e-06 6.76734498e-07 1.45855678e-06
 2.50068291e-04]
not reaching the requested tolerance 4.187226295471191e-06.
Use iteration 19 instead with accuracy 
4.346425155082264e-05.

  _, diffusion_map = lobpcg(laplacian, X, M=M, tol=tol, largest=False)
/home/nic/repos/movpasd/pawncounter/.venv/lib/python3.14/site-packages/sklearn/manifold/_spectral_embedding.py:430: UserWarning: Exited postprocessing with accuracies 
[2.41942092e-16 1.69419626e-06 6.83473186e-07 1.45757598e-06
 2.13486012e-04]
not reaching the requested tolerance 4.187226295471191e-06.
  _, diffusion_map = lobpcg(laplacian, X, M=M, tol=tol, largest=False)


position_id,pos,x,y,z
u32,u128,f64,f64,f64
0,18963252907773419061505,-9.3731e-8,-1.9393e-8,-0.039049
1,18963252907773419061506,-0.018115,0.003353,-0.030406
2,18963252907773419062016,-0.000752,0.013993,-0.029499
3,18963252907773419061504,-0.008202,0.007456,-0.033412
4,18963252907773419061761,0.018115,-0.003353,-0.030406
…,…,…,…,…
276,4740813226943354766338,-0.014675,-0.012466,0.038346
277,4740813226943354766848,-0.005064,-0.009999,0.038992
278,4759259971017064317956,0.014675,-0.012466,0.038346


In [180]:
_DEPTH = 3

_TRANSITION_COLOURS = {
    "domain": ["A", "CK", "CQ", "R"],
    "range": ["firebrick", "green", "aqua", "grey"],
}

_edges_df = (
    transitions.lazy()
    .filter(pl.col("transition_depth") < _DEPTH)
    .join(
        spectral_embeddings.lazy().select(start_pos="pos", x0="x", y0="y", z0="z"),
        on="start_pos",
    )
    .join(
        spectral_embeddings.lazy().select(end_pos="pos", x1="x", y1="y", z1="z"),
        on="end_pos",
    )
    .collect()
)

(
    alt.Chart(_edges_df)
    .mark_rule(strokeWidth=5, opacity=0.3)
    .encode(
        alt.X("x0"),
        alt.X2("x1"),
        alt.Y("y0"),
        alt.Y2("y1"),
        alt.Color("transition_type:N").scale(**_TRANSITION_COLOURS),  # type: ignore
    )
    .properties(width=300, height=300)
)

alt.Chart(...)

In [181]:
@button("Render transition mesh")
def render_transition_mesh():
    # Line mesh of transitions near the start, coloured by transition_type. Like the
    # Altair cell, only keep transitions whose source is within _DEPTH BFS levels.
    _DEPTH = 3

    # pos -> global position_id, and the embedding coordinates (row k == position_id k).
    _id = spectral_embeddings.select("pos", "position_id")
    _emb = spectral_embeddings.sort("position_id").select("x", "y", "z").to_numpy()

    # Edges: depth-filtered transitions as global position_id index pairs.
    _edges_df = (
        transitions.lazy()
        .filter(pl.col("transition_depth") < _DEPTH)
        .join(
            _id.lazy().rename({"pos": "start_pos", "position_id": "i"}), on="start_pos"
        )
        .join(_id.lazy().rename({"pos": "end_pos", "position_id": "j"}), on="end_pos")
        .collect()
    )
    _edges = _edges_df.select("i", "j").to_numpy()

    _TRANSITION_COLOURS = {
        "A": (0.70, 0.13, 0.13),  # firebrick
        "CK": (0.00, 0.50, 0.00),  # green
        "CQ": (0.00, 1.00, 1.00),  # aqua
        "R": (0.50, 0.50, 0.50),  # grey
    }
    _edge_colours = np.array(
        [_TRANSITION_COLOURS[t] for t in _edges_df["transition_type"].cast(pl.String)]
    )

    # The initial state (BFS root) is the start_pos of the depth-0 transitions.
    _initial_pos = init_position()
    _initial_pos_int = int(_initial_pos[0]) + 2**64 * int(_initial_pos[1])
    _initial_id = spectral_embeddings.filter(pl.col("pos") == _initial_pos_int)[
        "position_id"
    ].item()

    display_coloured_mesh(
        _emb,
        _edges,
        _edge_colours,
        name="transitions",
        highlight_nodes=[_initial_id],  # type: ignore
    )


render_transition_mesh()


Button(description='Render transition mesh', style=ButtonStyle())